# Multimodal Meme Understanding: Image, Text, or Both?
### Deep Learning Final Project
Dataset: Memotion 7k (SemEval-2020 Task 8, Task A: 3-class sentiment).

Pipeline (see `docs/project-roadmap.md`):
1. Environment setup & Google Drive mount
2. Dataset download & verification
3. Data preparation & preprocessing
4. Text-only Baseline (BERT) - Role B
5. Image-only Baseline (ResNet50) - Role C
6. Multimodal Fusion (Concat & Cross-Attention) - Role D
7. Evaluation, ablation & error analysis - Role E & F
8. Demo inference


## 1. Environment Setup & Google Drive Mount

In [12]:
from google.colab import drive
drive.mount('/content/drive')
import os

BRANCH = os.environ.get('BRANCH', 'main')
REPO_URL = 'https://github.com/arcrek/dl2026.git'
ROOT = '/content/drive/MyDrive/dl2026_memotion'
os.environ['DATA_DIR'] = '/content/data/memotion'
os.environ['FEATURE_DIR'] = f'{ROOT}/features'
os.environ['RESULTS_DIR'] = f'{ROOT}/results'

for d in ('features', 'results'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

if os.path.exists('/content/repo'):
    !git -C /content/repo fetch && git -C /content/repo checkout $BRANCH && git -C /content/repo pull
else:
    !git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo

!pip install -q transformers scikit-learn pyarrow huggingface_hub pytest pandas


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/repo


## 2. Dataset Download & Verification
> **Note:** For the Text-only Baseline (Role B), you can pass `--jsonl-only` to `scripts/download_data.sh` to download text metadata in seconds without fetching 1.5GB of images.

In [13]:
TAR = f'{ROOT}/memotion.tar'
if os.path.exists(TAR) and not os.path.exists('/content/data/memotion/train.jsonl'):
    !mkdir -p /content/data && tar -xf "$TAR" -C /content/data

!bash scripts/download_data.sh
!python src/data.py --images

if not os.path.exists(TAR):
    !tar -cf "$TAR" -C /content/data memotion


train.jsonl already exists, skipping download...
validation.jsonl already exists, skipping download...
test.jsonl already exists, skipping download...
Dataset ready in /content/data/memotion
train       n= 5593 labels={0: 518, 1: 1762, 2: 3313} OK
  All 5593 images verified for train.
validation  n=  699 labels={0: 59, 1: 221, 2: 419} OK
  All 699 images verified for validation.
test        n=  700 labels={0: 54, 1: 218, 2: 428} OK
  All 700 images verified for test.
Holdout: fit=5034 holdout=559 dist={0: 54, 1: 165, 2: 340}


## 3. Data Preparation & Preprocessing

### Label Normalization
Raw sentiment labels in Memotion 7k follow a 5-point scale:
`very_positive`, `positive`, `neutral`, `negative`, `very_negative`

For standard 3-class sentiment classification, they are consolidated into:
- `positive` (from very positive, positive, pos)
- `neutral` (from neutral, neu)
- `negative` (from very negative, negative, neg)


In [ ]:
import os, sys
sys.path.insert(0, "src")
import pandas as pd
from PIL import Image
from preprocessing import (
    normalize_sentiment,
    is_valid_image,
    get_image_path,
    load_raw_memotion_df,
    LABEL2ID,
    ID2LABEL,
)

def normalize_sentiment(label):
    if pd.isna(label):
        return None

    label = str(label).strip().lower()
    label = label.replace("_", " ")

    if label in ["very positive", "positive", "pos"]:
        return "positive"
    if label in ["neutral", "neu"]:
        return "neutral"
    if label in ["very negative", "negative", "neg"]:
        return "negative"

    return None


### 3.1. Create Clean DataFrame
After this step, the dataset contains only 3 primary columns:
`text`, `image_path`, `label_name`

Samples missing valid image files or sentiment labels are dropped.


In [ ]:
# Load DataFrame (supports Kaggle df, Colab, or local jsonl)
if "df" not in locals():
    df = load_raw_memotion_df(os.environ.get("DATA_DIR", "data/memotion"))

data = pd.DataFrame()
text_col = "text_ocr" if "text_ocr" in df.columns else ("text_corrected" if "text_corrected" in df.columns else "text")
image_col = "image_name" if "image_name" in df.columns else ("img" if "img" in df.columns else "image")
sentiment_col = "overall_sentiment" if "overall_sentiment" in df.columns else ("raw_sentiment" if "raw_sentiment" in df.columns else "label")

data["text"] = df[text_col].fillna("").astype(str)
data["image_path"] = df[image_col].apply(lambda x: get_image_path(x, base_dir=os.environ.get("DATA_DIR", "data/memotion")))
data["label_name"] = df[sentiment_col].apply(normalize_sentiment)

print("Before filtering invalid samples:", len(data))
print("Samples with found images:", data["image_path"].notna().sum())
print("Samples with valid labels:", data["label_name"].notna().sum())

# Drop samples missing image or label
data = data.dropna(subset=["image_path", "label_name"]).reset_index(drop=True)

print("After dropping missing image/label:", len(data))
print(data["label_name"].value_counts())
data.head()


### 3.2. Filter Corrupted Images Before Training
Some images in Memotion 7k may be corrupted or truncated. If not filtered beforehand, the DataLoader may crash during training with:
```
OSError: image file is truncated
```
Therefore, each image file is verified once before training.


In [ ]:
def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            img.convert("RGB")

        return True
    except Exception:
        return False

before_filter = len(data)
data = data[data["image_path"].apply(is_valid_image)].reset_index(drop=True)
after_filter = len(data)

print("Corrupted images dropped:", before_filter - after_filter)
print("Final sample count for training:", after_filter)
print(data["label_name"].value_counts())
data.head()


### 3.3. Encode Labels to Numeric IDs
PyTorch models require numeric target labels rather than strings:
- `negative`: 0
- `neutral`: 1
- `positive`: 2


In [ ]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

data["label"] = data["label_name"].map(label2id)
data.head()


## 4. Text-only Baseline (BERT) - Role B
Fine-tune `bert-base-uncased` with Differential Learning Rate across seeds 0, 1, 2.
The training module is decoupled into 4 distinct functions:
1. Loss & Optimizer configuration (`build_loss_and_optimizer`)
2. Evaluation function (`evaluate`)
3. Single-epoch training step (`train_epoch`)
4. Multi-epoch training loop (`train_loop`)

### 4.1 Loss Function & Optimizer Setup

In [15]:
import sys
sys.path.insert(0, 'src')
import torch
import torch.nn as nn
from text import BertMemeClassifier, build_loss_and_optimizer

print('Part 1: Loss & Optimizer module loaded.')


Part 1: Loss & Optimizer module loaded.


### 4.2 Evaluation Function

In [16]:
from text import evaluate

print('Part 2: Evaluation module loaded.')


Part 2: Evaluation module loaded.


### 4.3 Training Step (Single Epoch)

In [17]:
from text import train_epoch

print('Part 3: Single-epoch training step module loaded.')


Part 3: Single-epoch training step module loaded.


### 4.4 Training Loop & Execution

In [18]:
from text import train_loop

print('Part 4: Multi-epoch training loop module loaded.')


Part 4: Multi-epoch training loop module loaded.


In [19]:
!python src/text.py --seeds 0 1 2 --epochs 5 --batch 32


Starting Text Baseline training on device: CUDA
Config: {'epochs': 5, 'batch': 32, 'lr_backbone': 1.5e-05, 'lr_head': 0.0005, 'max_length': 128, 'workers': 2, 'dropout': 0.2, 'weight_decay': 0.01}
Seeds: [0, 1, 2]

==================== Training Seed 0 ====================
Loading weights: 100% 199/199 [00:00<00:00, 4752.82it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading f

In [20]:
!pytest tests/test_results.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/repo
plugins: langsmith-0.14.2, anyio-4.15.1, platformdirs-4.12.2, typeguard-4.6.0
collected 3 items                                                              

tests/test_results.py::test_roundtrip PASSED                             [ 33%]
tests/test_results.py::test_rejects_bad_probs_and_lengths PASSED         [ 66%]
tests/test_results.py::test_config PASSED                                [100%]

============================== 3 passed in 0.13s ===============================


In [21]:
import os, sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
from results import load_results

records = []
for s in [0, 1, 2]:
    try:
        res = load_results('text', s)
    except FileNotFoundError:
        continue
    records.append({
        'Seed': s,
        'Best Epoch (Holdout)': res['extra'].get('best_epoch', 'N/A'),
        'Holdout Macro-F1': round(res['extra'].get('holdout_macro_f1', 0.0), 4),
        'Test Accuracy': round(res['accuracy'], 4),
        'Test Macro-F1': round(res['macro_f1'], 4)
    })

if records:
    df = pd.DataFrame(records)
    if len(records) > 1:
        mean_row = pd.DataFrame([{
            'Seed': 'Mean +/- Std',
            'Best Epoch (Holdout)': '-',
            'Holdout Macro-F1': f"{df['Holdout Macro-F1'].mean():.4f} +/- {df['Holdout Macro-F1'].std():.4f}",
            'Test Accuracy': f"{df['Test Accuracy'].mean():.4f} +/- {df['Test Accuracy'].std():.4f}",
            'Test Macro-F1': f"{df['Test Macro-F1'].mean():.4f} +/- {df['Test Macro-F1'].std():.4f}"
        }])
        summary_df = pd.concat([df, mean_row], ignore_index=True)
    else:
        summary_df = df
    display(summary_df)
else:
    print('No results found. Run training cell above first.')


,Seed,Best Epoch (Holdout),Holdout Macro-F1,Test Accuracy,Test Macro-F1
0,0,2,0.3337,0.3914,0.3283
1,1,5,0.2942,0.3543,0.2988
2,2,1,0.3426,0.55,0.3677
3,Mean ± Std,-,0.3235 ± 0.0258,0.4319 ± 0.1039,0.3316 ± 0.0346



=== SUMMARY RESULT FOR SECTION 5 REPORT ===
Text-only (BERT) | Accuracy: 0.4319 ± 0.1039 | Macro-F1: 0.3316 ± 0.0346


In [22]:
try:
    from google.colab import files
    for s in [0, 1, 2]:
        f_path = f"results/text_seed{s}.json"
        if os.path.exists(f_path):
            files.download(f_path)
except Exception:
    pass


## 5. Image-only Baseline (ResNet50)


## 6. Multimodal Fusion (Concat & Cross-Attention)


## 7. Evaluation, Ablation & Error Analysis


## 8. Demo Inference
